# Wine quality data exploration

This notebook walks through the same data path used by the training pipeline: download, schema validation, cleaning, binary target creation, stratified splitting, and train-only standardization.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

from src.data import (
    FEATURE_COLUMNS,
    clean_and_label,
    download_dataset,
    load_raw_data,
    split_dataset,
    summarize_dataset,
)

data_dir = Path("../data/raw")
report_dir = Path("../reports/notebooks")
data_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
raw_path = download_dataset(data_dir / "winequality-red.csv")
raw_frame = load_raw_data(raw_path)
cleaned_frame = clean_and_label(raw_frame, quality_threshold=6)

{
    "raw_rows": len(raw_frame),
    "cleaned_rows": len(cleaned_frame),
    "features": len(FEATURE_COLUMNS),
}


In [ ]:
data_summary = summarize_dataset(cleaned_frame)
pd.Series(data_summary["target_counts"], name="rows")


In [ ]:
axis = cleaned_frame["quality"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(7, 4),
)
axis.set_title("Original wine quality score distribution")
axis.set_xlabel("quality score")
axis.set_ylabel("rows")
axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(report_dir / "quality_distribution.png", dpi=160)


In [ ]:
train_frame, validation_frame, test_frame = split_dataset(cleaned_frame, seed=42)

split_summary = []
for name, frame in [
    ("train", train_frame),
    ("validation", validation_frame),
    ("test", test_frame),
]:
    split_summary.append(
        {
            "split": name,
            "rows": len(frame),
            "positive_rate": frame["good_quality"].mean(),
            "mean_quality": frame["quality"].mean(),
        }
    )

split_frame = pd.DataFrame(split_summary)
split_frame


In [ ]:
scaler = StandardScaler().fit(train_frame[FEATURE_COLUMNS])
scaled_train = pd.DataFrame(
    scaler.transform(train_frame[FEATURE_COLUMNS]),
    columns=FEATURE_COLUMNS,
)

scaling_check = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "scaled_train_mean": scaled_train.mean().round(4).to_numpy(),
        "scaled_train_std": scaled_train.std(ddof=0).round(4).to_numpy(),
    }
)
scaling_check


In [ ]:
correlations = cleaned_frame[FEATURE_COLUMNS + ["quality"]].corr()["quality"].drop("quality")
top_correlations = correlations.abs().sort_values(ascending=False).head(8).index
correlation_frame = correlations.loc[top_correlations].sort_values()

axis = correlation_frame.plot(kind="barh", figsize=(7, 4))
axis.set_title("Feature correlation with original quality score")
axis.set_xlabel("Pearson correlation")
axis.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(report_dir / "quality_correlations.png", dpi=160)
correlation_frame


In [ ]:
split_frame.to_csv(report_dir / "split_summary.csv", index=False)
scaling_check.to_csv(report_dir / "scaling_check.csv", index=False)
report_dir
